In [37]:
print(all_embeddings.shape)

torch.Size([2324, 512])


In [39]:
query= np.random.rand(1, 512).astype("float32") #( how many queries =1, vector size=512 )
distances, indices = index.search(query, k=3)
print(distances)
print(indices)
#indices	=nearest vector positions
#distances=	how close they are


[[176.91309 177.31801 177.31801]]
[[1141  988 1091]]


In [ ]:
import faiss
dimension= 512 #CLIP outputs 512-length vectors
index= faiss.IndexFlatL2(dimension) #A vector database using L2 distance (Euclidean distance)
#Facebook AI Similarity Search
index.add(all_embeddings)


In [35]:
!pip install faiss-cpu

  Using cached faiss_cpu-1.13.2-cp310-cp310-win_amd64.whl.metadata (7.6 kB)
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   -----------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   --------------------------- ------------ 12.8/18.9 MB 83.2 kB/s eta 0:01:13
   --------------------------- ------------ 12.8/18.9 MB 83.2 kB/s eta 0:01:13
   --------------------------- ------------ 12.8/18.9 MB 83.2 kB/s eta 0:01:13
   --------------------------- ------------ 12.8/18.9 MB 83.2 kB/s eta 0:01:13
   --------------------------- ------------ 12.8/18.9 MB 83.2 kB/s eta 0:01:13
   --------------------------- ------------ 13.1/18.9 MB 87.6 kB/s eta 0:01:06
   --------------------------- ------------ 13.1/18.9 MB 87.6 kB/s eta 0:01:06
   --------------------------- ------------ 13.1/18.9 MB 87.6 kB/s eta 0:01:06
   --------------------------- ------------ 13.1/18.9 MB 87.6 kB/s eta 0:01:06
   --------------------------- ------------ 13.1/18.9 MB 87.6 kB/s eta 0:01:06
   --------------------------- ------------ 13.1/18.9 MB 87.6 kB/s eta 0:01:06
   --------------------------- ------------ 13.1/18.9 MB 87.6 kB/s eta 0:01:06
   --------------------------- ------------ 13.1/18.

In [34]:
torch.save(all_embeddings, "image_embeddings.pt")
torch.save(all_labels, "image_labels.pt")
print("💾 Embeddings and labels saved!")

💾 Embeddings and labels saved!


In [33]:

from tqdm import tqdm
all_embeddings = []
all_labels = []

for path, label in tqdm(zip(image_paths, labels), total=len(image_paths)):
    try:
        # Load image
        image = Image.open(path).convert("RGB")  # ensure RGB format
        # Preprocess (resizing, normalization)
        image_input = preprocess(image).unsqueeze(0).to(device)
        
        # Get embedding
        with torch.no_grad():
            embedding = model.encode_image(image_input)
            embedding = embedding / embedding.norm(dim=-1, keepdim=True)  # normalize

        all_embeddings.append(embedding.cpu())
        all_labels.append(label)
    except Exception as e:
        print(f"⚠️ Skipping {path}: {e}")

# Convert list to tensor
all_embeddings = torch.cat(all_embeddings, dim=0)
print(f"✅ All embeddings shape: {all_embeddings.shape}")

100%|██████████| 2324/2324 [11:40<00:00,  3.32it/s]

✅ All embeddings shape: torch.Size([2324, 512])


In [24]:
image_paths = []
labels = []

for category in os.listdir(dataset_dir):
    category_path = os.path.join(dataset_dir, category)
    if not os.path.isdir(category_path):
        continue

    for dest in os.listdir(category_path):
        dest_path = os.path.join(category_path, dest)
        if not os.path.isdir(dest_path):
            continue

        for img_file in os.listdir(dest_path):
            if img_file.lower().endswith((".jpg", ".jpeg", ".png")):
                image_paths.append(os.path.join(dest_path, img_file))
                labels.append(dest)  # innermost folder name as label

print(f"✅ Total images found: {len(image_paths)}")
print(f"Example labels: {set(labels)}")

✅ Total images found: 2324
Example labels: {'matara_city', 'anuradhapura_ruins', 'mount_lavinia_beach', 'jaffna_city', 'trincomalee_beach', 'nuwara_eliya', 'galle_city', 'badulla_city', 'polonnaruwa_ruins', 'isurumuniya_temple', 'sigiriya', 'knuckles_mountain_range', 'wilpattu_national_park', 'haputale', 'ella', 'talpe_beach', 'unawatuna_beach', 'colombo_city', 'gal_oya_national_park', 'yapahuwa_rock_fortress', 'tissamaharama_temple', 'kaudulla_national_park', 'negombo_city', 'aluvihare_rock_temple', 'hikkaduwa_beach', 'lipton_seat', 'arugam_bay', 'mihintale', 'anuradhapura_city', 'pasikuda_beach', 'galle_fort', 'wasgamuwa_national_park', 'nilaveli_beach', 'sinharaja_rainforest', 'kahandamodara_beach', 'matara_beach', 'mirissa_beach', 'kurunegala_city', 'udawalawe_national_park', 'horton_plains', 'bandarawela', 'waikkal_beach', 'dambulla_cave_temple', 'yala_national_park', 'badulla', 'kalpitiya_beach', 'trincomalee_city', 'kandy', 'meemure_village', 'bundala_national_park', 'adams_peak

In [ ]:
import clip
model,preprocess= clip.load('ViT-B/32', device= device) # clip model, preprocess is a function tht directly convert images into vectors,It does resizing, cropping, normalizing pixel values, etc
model.eval()
#eval =This is required when you just want to generate embeddings, not train the model.
#Turns off training features like dropout or gradient updates.
#ViT = Vision Transformer (the part that processes images)

#B = Base size (smaller model than Large, faster)

#32 = patch size (how the image is split for processing)

100%|███████████████████████████████████████| 338M/338M [01:57<00:00, 3.02MiB/s]


CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): Sequential(
        (0): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): QuickGELU()
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
        (1): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          

In [ ]:
!pip install torch 
!pip install git+https://github.com/openai/CLIP.git
!pip install faiss-cpu



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Cloning https://github.com/openai/CLIP.git to c:\users\admin\appdata\local\temp\pip-req-build-z0ljkmgz
  Resolved https://github.com/openai/CLIP.git to commit ded190a052fdf4585bd685cee5bc96e0310d2c93
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---

  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git 'C:\Users\ADMIN\AppData\Local\Temp\pip-req-build-z0ljkmgz'
  DEPRECATION: Building 'clip' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'clip'. Discussion can be found at https://github.com/pypa/pip/issues/6334

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached faiss_cpu-1.13.2-cp310-cp310-win_amd64.whl.metadata (7.6 kB)
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   -----------

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\ADMIN\\AppData\\Local\\Temp\\pip-unpack-ffdhs13i\\faiss_cpu-1.13.2-cp310-cp310-win_amd64.whl'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



   --- ------------------------------------ 1.6/18.9 MB 16.8 kB/s eta 0:17:09
   --- ------------------------------------ 1.6/18.9 MB 16.8 kB/s eta 0:17:09
   --- ------------------------------------ 1.6/18.9 MB 16.8 kB/s eta 0:17:09
   --- ------------------------------------ 1.6/18.9 MB 16.8 kB/s eta 0:17:09
   --- ------------------------------------ 1.6/18.9 MB 16.8 kB/s eta 0:17:09
   --- ------------------------------------ 1.6/18.9 MB 16.8 kB/s eta 0:17:09
   --- ------------------------------------ 1.6/18.9 MB 16.8 kB/s eta 0:17:09
   --- ------------------------------------ 1.6/18.9 MB 16.8 kB/s eta 0:17:09
   --- ------------------------------------ 1.6/18.9 MB 16.8 kB/s eta 0:17:09
   --- ------------------------------------ 1.6/18.9 MB 16.8 kB/s eta 0:17:09
   --- ------------------------------------ 1.6/18.9 MB 16.8 kB/s eta 0:17:09
   --- ------------------------------------ 1.6/18.9 MB 16.8 kB/s eta 0:17:09
   --- ------------------------------------ 1.6/18.9 MB 16.8 kB

   --- ------------------------------------ 1.8/18.9 MB 15.1 kB/s eta 0:18:47
   --- ------------------------------------ 1.8/18.9 MB 15.1 kB/s eta 0:18:47
   --- ------------------------------------ 1.8/18.9 MB 15.1 kB/s eta 0:18:47
   --- ------------------------------------ 1.8/18.9 MB 15.1 kB/s eta 0:18:47
   --- ------------------------------------ 1.8/18.9 MB 15.1 kB/s eta 0:18:47
   --- ------------------------------------ 1.8/18.9 MB 15.1 kB/s eta 0:18:47
   --- ------------------------------------ 1.8/18.9 MB 15.1 kB/s eta 0:18:47
   --- ------------------------------------ 1.8/18.9 MB 15.1 kB/s eta 0:18:47
   --- ------------------------------------ 1.8/18.9 MB 15.1 kB/s eta 0:18:47
   --- ------------------------------------ 1.8/18.9 MB 15.1 kB/s eta 0:18:47
   --- ------------------------------------ 1.8/18.9 MB 15.1 kB/s eta 0:18:47
   --- ------------------------------------ 1.8/18.9 MB 15.1 kB/s eta 0:18:47
   --- ------------------------------------ 1.8/18.9 MB 15.1 kB/

In [12]:
import torch
device= 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device : {device}")


device : cpu


In [8]:
import tensorflow as tf
dataset_dir= r"./dataset"
dataset= tf.keras.utils.image_dataset_from_directory(dataset_dir)


Found 2324 files belonging to 5 classes.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt